In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpfull assistant"
)

In [ ]:
resp=agent.invoke(input={"message":[
    {"role":"user","content":"Je m'appelle Abdo"}
]})

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
resp=agent.invoke(input={"message":[
    {"role":"user","content":"Comment je m'appelle?"}
]})

In [ ]:
from langchain.agent.middleware import ModelRequest, ModelResponse, wrap_model_call
from lanfchain.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temerature=0)
advenced_llm = ChatOpenAI(model="gpt-4o", temerature=0)

In [ ]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, handler)->ModelResponse:
    env = request.runtime.context.get("env","test")
    if env == "prod":
        model = advenced_llm
    else :
        model = basic_llm
    return handler(request.override(model=model))

In [ ]:
agent2 = create_agent(
    model=basic_llm,
    tools=[],
    middleware=[dynamic_model_selection]
)

In [ ]:
resp = agent2.invoke(
    input={"messages":[HumanMessage("C'est quoi un agent AI")]},
    context={"env":"prod"}
    )

In [ ]:
from IPython.display import Markdown

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
memory = InMemorySaver()
agent3 = create_agent(
    model = "openai:gpt'4o",
    system_prompt="You are a helpfull asistant",
    checkpointer = memory
)

In [ ]:
config = {"configurable":{"thread_id":1}}
resp = agent3.invoke(
    input={"messages":[HumanMessage("Je m'appelle Abdo")]},
    config=config
    )

In [ ]:
print(resp["messages"][-1].content)

In [ ]:
resp = agent3.invoke(
    input={"messages":[HumanMessage("Comment je m'appelle?")]},
    config=config
    )

In [ ]:
print(resp["messages"][-1].content)

In [ ]:
from langchain.tools import tool

In [ ]:
@tool
def get_weather(city : str):
    """
    Get The weather of the given city
    """
    print(f"Weather Tool invoked")
    return {
        "city":city,
        "temperature":21,
        "humidity":77,
        "pressure":110
    }

In [ ]:
@tool
def get_employee_info(employee_name : str):
    """
    Get infos about the given employee
    """
    print(f"Info employee tool invoked")
    return {
        "name":employee_name,
        "salary":21000,
        "seniority":7
    }

In [ ]:
agent4 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info],
    checkpoint=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [ ]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(
    input={'messages':[HumanMessage("La météo à Rabat")]},
    config=config
)

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(
    input={'messages':[HumanMessage("Quel est le salaire de Hassan?")]},
    config=config
)

In [ ]:
print(resp['messages'][-1].content)

In [ ]:
load_dotenv(override=True)

In [ ]:
from ddgs import DDGS

In [ ]:
@tool
def web_searchDDGS(query: str, num_results:int=5) -> str:
    """
    Search the web usin DuckDuckGo
    Args:
    query : Search query string
    num_results : Number of results to return (Default=5)
    Returns:
    Formatted search results with titles, descptions and Urls
    """
    try:
        print(f'Search tool is called for {query}')
        ddgs_search = DDGS()
        results=ddgs_search.text(query=query, max_results=num_results, backend="google")
        if not results:
            return f"No results found for {query} "
        formatted_results = [f"Search for {query} : \n"]
        for i, result in enumerate(results,1):
            title = result.get("title","No Title")
            body = result.get("body","No description available")
            href = result.get("href","")
            formatted_results.append( f"{i}. **{title}**. {body}. {href}")
        return formatted_results
    except Exception as e:
        print(str(e))

In [ ]:
agent5 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info, web_searchDDGS],
    checkpoint=memory,
    system_prompt="answer the user question using only provided tools",
    debug=True
    )

In [ ]:
rep=agent5.invoke(input={"messages":[HumanMessage("Search for python tutorials")]},
              config=config
              )

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
tavily = TavilySearch(max_results=10, search_depth="advenced")

In [ ]:
@tool
def search_webTavily(query:str):
    """
    Search for general current web results
    """
    print(f"search_web invoked with {query}")
    results = tavily.invoke({"query":query})
    return results

In [ ]:
agent6 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info, search_webTavily],
    checkpoint=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [ ]:
rep=agent6.invoke(input={"messages":[HumanMessage("Atualités sur le maroc")]},
              config=config
              )

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

In [ ]:
from langchain_experimental.tools import PythonREPLTool

In [ ]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [ ]:
agent7 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""Générer et exécuter le code python
    en plançant le code python générer et le résultat  d'execution de ce code dans le fichier doc.txt
    """,
    debug=True
)

In [ ]:
resp=agent7.invoke(
    input={"message":[HumanMessage("Je veux trier les deux listes [6,8,2,1] et [5,1,9,4] et puis additionner les deux listes")]}
)

In [ ]:
print(resp['messages'][-1].content)